**Instalação do MPI4PY**

In [1]:
# 1. Instala o OpenMPI e bibliotecas de desenvolvimento
!apt-get install -y mpi-default-bin mpi-default-dev

# 2. Instala o mpi4py (wrapper para Python)
!pip install mpi4py

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
mpi-default-bin is already the newest version (1.14).
mpi-default-bin set to manually installed.
mpi-default-dev is already the newest version (1.14).
mpi-default-dev set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.1 MB/s eta 0:00:0000:01


***Hello Word***

In [2]:
%%writefile mpi_teste.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f"Olá do processo {rank} de um total de {size}!")

Writing mpi_teste.py


In [3]:
!mpirun --allow-run-as-root -np 10 --oversubscribe python mpi_teste.py

Olá do processo 9 de um total de 10!
Olá do processo 0 de um total de 10!
Olá do processo 6 de um total de 10!
Olá do processo 7 de um total de 10!
Olá do processo 1 de um total de 10!
Olá do processo 8 de um total de 10!
Olá do processo 4 de um total de 10!
Olá do processo 5 de um total de 10!
Olá do processo 2 de um total de 10!
Olá do processo 3 de um total de 10!


In [4]:
%%writefile exemplo_openmp.cpp
#include <iostream>
#include <omp.h>

int main() {
    // Define o número de threads (opcional, o padrão é o total de núcleos)
    omp_set_num_threads(4);

    #pragma omp parallel
    {
        int id = omp_get_thread_num();
        int total = omp_get_num_threads();
        printf("Olá da thread %d de um total de %d\n", id, total);
    }

    return 0;
}

Writing exemplo_openmp.cpp


In [5]:
# Compilação
!g++ -fopenmp exemplo_openmp.cpp -o programa_openmp

# Execução
!./programa_openmp

Olá da thread 3 de um total de 4
Olá da thread 2 de um total de 4
Olá da thread 0 de um total de 4
Olá da thread 1 de um total de 4


**Teste CACHE**

In [6]:
%%writefile teste_cache.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// No Colab, 10000x10000 de doubles consome cerca de 800MB de RAM.
#define MAX 10000

int main() {
    int i, j;
    struct timespec start, end;
    double time_row, time_col;

    // Alocação dinâmica para garantir que o Colab gerencie a memória no Heap
    double **A = (double **)malloc(MAX * sizeof(double *));
    for (i = 0; i < MAX; i++) A[i] = (double *)malloc(MAX * sizeof(double));
    double *x = (double *)malloc(MAX * sizeof(double));
    double *y = (double *)malloc(MAX * sizeof(double));

    // Inicialização dos dados
    for (i = 0; i < MAX; i++) {
        x[i] = i * 0.5;
        y[i] = 0;
        for (j = 0; j < MAX; j++) {
            A[i][j] = (i + j) * 0.1;
        }
    }

    printf("Iniciando testes com matriz %d x %d...\n\n", MAX, MAX);

    // --- FORMA 1: ACESSO POR LINHA (Eficiente / Cache Friendly) ---
    // Baseado no "First pair of loops" da image_765e51.jpg
    clock_gettime(CLOCK_MONOTONIC, &start);
    for (i = 0; i < MAX; i++) {
        for (j = 0; j < MAX; j++) {
            y[i] += A[i][j] * x[j];
        }
    }
    clock_gettime(CLOCK_MONOTONIC, &end);
    time_row = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("Tempo (Acesso por Linha - Sugerido): %f segundos\n", time_row);

    // Reset de y
    for (i = 0; i < MAX; i++) y[i] = 0;

    // --- FORMA 2: ACESSO POR COLUNA (Ineficiente / Cache Miss) ---
    // Baseado no "Second pair of loops" da image_765e51.jpg
    clock_gettime(CLOCK_MONOTONIC, &start);
    for (j = 0; j < MAX; j++) {
        for (i = 0; i < MAX; i++) {
            y[i] += A[i][j] * x[j];
        }
    }
    clock_gettime(CLOCK_MONOTONIC, &end);
    time_col = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("Tempo (Acesso por Coluna - Lento):     %f segundos\n", time_col);

    printf("\nO acesso por coluna foi %.2fx mais lento.\n", time_col / time_row);

    // Liberação de memória
    for (i = 0; i < MAX; i++) free(A[i]);
    free(A); free(x); free(y);

    return 0;
}

Writing teste_cache.c


In [7]:
!gcc -O0 teste_cache.c -o teste_cache  # -O0 desativa otimizações do compilador para ver o efeito real
!./teste_cache

Iniciando testes com matriz 10000 x 10000...

Tempo (Acesso por Linha - Sugerido): 0.414304 segundos
Tempo (Acesso por Coluna - Lento):     2.301837 segundos

O acesso por coluna foi 5.56x mais lento.


In [8]:
%%writefile mpi_soma.py
from mpi4py import MPI
import numpy as np
import math
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Configuração do vetor global (apenas no processo ROOT)
N = 100000000
vetor_global = None

if rank == 0:
    # Cria um vetor com números de 1 a 100 (Soma esperada = 5050)
    vetor_global = np.arange(1, N + 1, dtype=int)
    print(f"[ROOT] Iniciando processamento com {size} processos.", flush=True)

# --- DISTRIBUIÇÃO DOS DADOS (Scatter) ---
# Calcula o tamanho do pedaço de cada processo
local_n = N // size
resto = N % size

# Vetor local que armazenará o pedaço de cada processo
if rank < resto:
    local_size = local_n + 1
    start_idx = rank * local_size
else:
    local_size = local_n
    start_idx = rank * local_size + resto

end_idx = start_idx + local_size

# Distribuição manual via fatiamento para suportar qualquer tamanho de vetor
vetor_local = None
if rank == 0:
    for i in range(1, size):
        i_local_n = local_n + (1 if i < resto else 0)
        i_start = i * i_local_n + (0 if i < resto else resto)
        if i >= resto:
            i_start = i * local_n + resto
        comm.send(vetor_global[i_start:i_start+i_local_n], dest=i)
    vetor_local = vetor_global[start_idx:end_idx]
else:
    vetor_local = comm.recv(source=0)

# Cada processo calcula sua soma local inicial
soma_local = int(np.sum(vetor_local))

# Sincronização para iniciar os testes de tempo de forma justa
comm.Barrier()

# ==========================================
# ESTRATÉGIA 1: Naive (Root centraliza tudo)
# ==========================================
t_start_naive = time.perf_counter()

if rank == 0:
    soma_final_naive = soma_local
    for i in range(1, size):
        parcial = comm.recv(source=i, tag=11)
        soma_final_naive += parcial
else:
    comm.send(soma_local, dest=0, tag=11)

comm.Barrier()
if rank == 0:
  t_end_naive = time.perf_counter()
  tempo_naive = t_end_naive - t_start_naive


# ==========================================
# ESTRATÉGIA 2: Árvore Binária
# ==========================================
comm.Barrier()
t_start_tree = time.perf_counter()

soma_arvore = soma_local
passos = int(math.log2(size))

# Loop baseado em divisões sucessivas por 2 (log2)
for passo in range(passos):
    divisor = 2 ** (passo + 1)
    sub_passo = 2 ** passo

    # Verifica se o processo é o acumulador (par no nível atual) ou o emissor (ímpar)
    if rank % divisor == 0:
        # Nó receptor recebe do seu par da direita (rank + sub_passo)
        origem = rank + sub_passo
        if origem < size:
            parcial_recebida = comm.recv(source=origem, tag=22)
            soma_arvore += parcial_recebida
    elif rank % sub_passo == 0:
        # Nó emissor envia para o seu acumulador da esquerda (rank - sub_passo)
        destino = rank - sub_passo
        comm.send(soma_arvore, dest=destino, tag=22)
        # Uma vez que enviou, o nó sai da árvore de computação nas próximas iterações
        break

comm.Barrier()
if rank == 0:
    t_end_tree = time.perf_counter()
    tempo_tree = t_end_tree - t_start_tree

    # Exibição dos Resultados Consolidados
    print("\n" + "="*40)
    print("      RESULTADOS COMPARTILHADOS", flush=True)
    print("="*40)
    print(f"Estratégia 1 (Naive):  Soma = {soma_final_naive} | Tempo = {tempo_naive:.7f}s", flush=True)
    print(f"Estratégia 2 (Árvore): Soma = {soma_arvore} | Tempo = {tempo_tree:.7f}s", flush=True)
    print("="*40)

Writing mpi_soma.py


In [9]:
!mpirun --allow-run-as-root -n 4 --oversubscribe python -u mpi_soma.py

[ROOT] Iniciando processamento com 4 processos.

      RESULTADOS COMPARTILHADOS
Estratégia 1 (Naive):  Soma = 5000000050000000 | Tempo = 0.0002374s
Estratégia 2 (Árvore): Soma = 5000000050000000 | Tempo = 0.0001285s


In [22]:
%%writefile mpi_hello.c
#include <stdio.h>
#include <string.h> /* For strlen */
#include <mpi.h> /* For MPI functions, etc */

const int MAX_STRING = 100;

int main(void) {
  char greeting[MAX_STRING];
  int comm_sz; /* Number of processes */
  int my_rank; /* My process rank */

  MPI_Init(NULL, NULL);
  MPI_Comm_size(MPI_COMM_WORLD, &comm_sz);
  MPI_Comm_rank(MPI_COMM_WORLD, &my_rank);

  //printf("Oi do processo %d\n", my_rank);

  if (my_rank != 0) {
    sprintf(greeting, "Greetings from process %d of %d!",
            my_rank, comm_sz);
    MPI_Send(greeting, strlen(greeting)+1, MPI_CHAR, 0, 0,
             MPI_COMM_WORLD);
  } else {
    printf("Greetings from process %d of %d!\n\n", my_rank,
           comm_sz);
    for (int q = 1; q < comm_sz; q++) {

    MPI_Recv(greeting,
             MAX_STRING,
             MPI_CHAR,
             MPI_ANY_SOURCE,
             0,
             MPI_COMM_WORLD,
             MPI_STATUS_IGNORE);

    printf("%s\n\n", greeting);
}
  }
  MPI_Finalize();
  return 0;
} /* main */

Overwriting mpi_hello.c


In [23]:
!mpicc -g -Wall -o mpi_hello mpi_hello.c

In [24]:
!mpiexec --allow-run-as-root -n 4 --oversubscribe ./mpi_hello

Greetings from process 0 of 4!

Greetings from process 3 of 4!

Greetings from process 1 of 4!

Greetings from process 2 of 4!

